# Fit Models

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier

from tqdm_joblib import tqdm_joblib
from tqdm.auto import tqdm

from bioacoustics.data import load_results
from bioacoustics.metrics import evaluate_multilabel_model

from bioacoustics.modeling import FitMode, LabelType
from bioacoustics.modeling import split_data

from bioacoustics.modeling import ignore_warnings
ignore_warnings() # TODO: fix the reasons for these warnings

%load_ext autoreload
%autoreload 2

/Users/vlad/Documents/University/Master-MIND/bioacoustic-species-detection/.venv/lib/python3.13/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Problems to think about:

- Multi-label prediction -> hierarchical models? implement with `Pipeline`
- Class imbalance -> tinker with class weights, adjust probabilities?
- Use additional metadata from train, not available in soundscapes
- Maybe use hybrid approaches? learn about how to combine different models' predictions

think about chunking the audio, tuning the threshold for ech class

## Load features and labels

In [53]:
data_train = load_results("features", "data_train", frozen=True)
data_train_soundscapes = load_results("features", "data_train_soundscapes", frozen=True)

Note that it doesn't make much sense to validate on iNat or XC data, since it's of different format than test soundscapes.

In [48]:
FIT_MODE = FitMode.SOUNDSCAPE_TO_SOUNDSCAPE
LABEL_TYPE = LabelType.CLASS
X_train, X_test, y_train, y_test = split_data(
    data_train,
    data_train_soundscapes,
    FIT_MODE,
    LABEL_TYPE,
    test_size=0.2,
    random_state=42,
)

## Predict class

Start with predicting class only, we may after use this result when predicting primary label (hierarchical approach).



In [49]:
# clf = LogisticRegression(solver="liblinear", max_iter=1000, class_weight="balanced")
# clf = RandomForestClassifier(
#     n_estimators=200, max_depth=None, n_jobs=-1, class_weight="balanced"
# )
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
)

In [50]:
pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),  # otherwise MFCC and RMS are incomparable
        (
            "clf",
            OneVsRestClassifier(clf, n_jobs=-2, verbose=2),
        ),
    ]
)
with tqdm_joblib(
    tqdm(
        desc="Training OvR", total=y_train.shape[1]
    )
):
    pipeline.fit(X_train, y_train)

Training OvR:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=-2)]: Done   3 out of   5 | elapsed:    2.2s remaining:    1.5s
[Parallel(n_jobs=-2)]: Done   5 out of   5 | elapsed:    2.8s finished


In [51]:
print(f"{FIT_MODE.name} predicting {LABEL_TYPE.name}".center(60))
y_pred, y_proba = evaluate_multilabel_model(pipeline, X_test, y_test)

         SOUNDSCAPE_TO_SOUNDSCAPE predicting CLASS          

================= CLASSIFICATION REPORT =================
              precision    recall  f1-score   support

    Amphibia       1.00      0.98      0.99       222
        Aves       0.96      0.68      0.79       136
     Insecta       1.00      0.98      0.99        96
    Mammalia       0.00      0.00      0.00        12
    Reptilia       0.50      0.33      0.40         6

   micro avg       0.99      0.86      0.92       472
   macro avg       0.69      0.59      0.63       472
weighted avg       0.96      0.86      0.90       472
 samples avg       0.97      0.90      0.92       472


================= THRESHOLD-BASED METRICS =================
Macro F1:   0.6347
Micro F1:   0.9186
Hamming loss: 0.0453

================= RANKING & PROBABILITY METRICS =================
Macro ROC AUC: 0.8701
Micro ROC AUC: 0.9792
Macro AP:      0.7099
Micro AP:      0.9720
LRAP:          0.9923


## Interpret the model

In [39]:
# TODO: get feature importance

# SANDBOX

In [40]:
# TODO: try XGBoost, RF etc, change multi-label strategies
# TODO: hierarchical approach - train a model per class
# and predict species conditioned on class (can try soft versions - multiply probas)

# TODO: alternative to hierarchical approach - include predicted class as a feature
# for species prediction
# TODO: the same idea for training metadata that is absent for the test
# - we can learn to predict this metadata as a secondary task and then include
# it in the model
# NOTE: attention to data leak between train - validation when using secondary tasks
# to generate features (class, metadata)

# TODO: can use this metadata as well for stratification - better split validation
# (make sure that there is no data leak because of the same site)
# TODO: check whether metadata alone can predict species (check for shortcut learning)

# TODO: smart cross validations

# TODO: quality of train audio from iNat and xeno-cant is poorer and further from test than train soundscapes,
#       so maybe we should use them only for species poorly covered by soundscapes

# TODO: impute some NaNs

# TODO: maybe stratify be species or sth else since train data had
#   too (really) many bird recordings, whereas soundscapes contain more amphibians

# TODO: maybe play with thresholds

# TODO: temporal smoothing of predicted probabilities!